## Load the CMIP6 data from api call into a prepared data array for visualization
The CMIP6 data is loaded to get cleaned warming rates, seasonal values, coordinates, and times into a clear data storage.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import gcsfs

# ============================================================
# CONFIGURATION
# ============================================================

MODEL = "GFDL-ESM4"
MEMBER = "r2i1p1f1"
GRID = None  # Set to a grid_label if multiple datasets match

START_YEAR = 1950
END_YEAR = 2014
BASELINE = (1951, 1980)
SEASONS = ["DJF", "MAM", "JJA", "SON"]

# ============================================================
# 1. FIND AND LOAD CMIP6 DATA
# ============================================================

catalog = pd.read_csv(
    "https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv"
)

matches = catalog[
    (catalog.activity_id == "CMIP") &
    (catalog.table_id == "Amon") &
    (catalog.variable_id == "tas") &
    (catalog.experiment_id == "historical") &
    (catalog.source_id == MODEL) &
    (catalog.member_id == MEMBER)
]

if GRID is not None:
    matches = matches[matches.grid_label == GRID]

if len(matches) != 1:
    raise ValueError(
        f"Expected one dataset, found {len(matches)}.\n"
        f"{matches[['grid_label', 'zstore']]}"
    )

fs = gcsfs.GCSFileSystem(token="anon")
store = fs.get_mapper(matches.iloc[0]["zstore"])

ds = xr.open_zarr(
    store,
    consolidated=True,
    decode_times=xr.coders.CFDatetimeCoder(use_cftime=True)
)

# Load the requested period, including December of the
# preceding year for the first complete winter.
tas = ds["tas"].sel(
    time=slice(
        f"{START_YEAR - 1}-12-01",
        f"{END_YEAR}-12-31"
    )
)

# ============================================================
# 2. ASSIGN SEASONS AND SEASON-YEARS
# ============================================================

time_values = tas.time.values

months = np.array([t.month for t in time_values])
years = np.array([t.year for t in time_values])

season_names = np.array([
    "DJF" if month in (12, 1, 2) else
    "MAM" if month in (3, 4, 5) else
    "JJA" if month in (6, 7, 8) else
    "SON"
    for month in months
])

# December belongs to the following winter year.
season_year = np.where(months == 12, years + 1, years)

tas = tas.assign_coords(
    season=("time", season_names),
    season_year=("time", season_year)
)

# ============================================================
# 3. CALCULATE SEASONAL MEAN TEMPERATURES
# ============================================================

seasonal_means = []

for season in SEASONS:
    selected = tas.where(tas.season == season, drop=True)

    # Identify complete seasons with three monthly records.
    counts = selected.time.groupby("season_year").count()
    complete_years = counts.season_year.where(
        counts == 3, drop=True
    )

    selected = selected.where(
        selected.season_year.isin(complete_years),
        drop=True
    )

    # Average the three months in each season-year.
    means = selected.groupby("season_year").mean("time")

    means = means.rename({"season_year": "year"})
    means = means.sel(year=slice(START_YEAR, END_YEAR))

    # Remove auxiliary coordinates before adding season dimension.
    means = means.drop_vars(
        ["season", "season_year"],
        errors="ignore"
    )

    seasonal_means.append(
        means.expand_dims(season=[season])
    )

# Combine seasons and convert Kelvin to Celsius.
temperature = xr.concat(
    seasonal_means, dim="season"
) - 273.15

temperature.name = "temperature"
temperature.attrs["units"] = "degC"

# ============================================================
# 4. CALCULATE TEMPERATURE ANOMALIES
# ============================================================

# Each season has its own baseline mean at each grid cell.
baseline = temperature.sel(
    year=slice(*BASELINE)
).mean("year")

anomaly = temperature - baseline
anomaly.name = "anomaly"
anomaly.attrs["units"] = "degC"

# ============================================================
# 5. CALCULATE SEASONAL WARMING RATES
# ============================================================

# Linear regression of seasonal temperature against year.
# Output is one warming rate per season and grid cell.
warming_rate = (
    temperature.polyfit(
        dim="year",
        deg=1,
        skipna=True
    )
    .polyfit_coefficients.sel(degree=1, drop=True)
    * 10
)

warming_rate.name = "warming_rate"
warming_rate.attrs["units"] = "degC/decade"

# ============================================================
# 6. COMBINE INTO ONE DATASET
# ============================================================

climate = xr.Dataset({
    "temperature": temperature.astype("float32"),
    "anomaly": anomaly.astype("float32"),
    "warming_rate": warming_rate.astype("float32")
})

climate.attrs.update({
    "model": MODEL,
    "member": MEMBER,
    "experiment": "historical",
    "start_year": START_YEAR,
    "end_year": END_YEAR,
    "baseline": f"{BASELINE[0]}-{BASELINE[1]}"
})

print(climate)

# ============================================================
# 7. SAVE PROCESSED DATA
# ============================================================

climate.to_netcdf("cmip6_processed.nc",engine="h5netcdf")

print("Saved cmip6_processed.nc")


<xarray.Dataset> Size: 109MB
Dimensions:       (season: 4, lat: 180, lon: 288, year: 65)
Coordinates:
  * season        (season) object 32B 'DJF' 'MAM' 'JJA' 'SON'
  * lat           (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 87.5 88.5 89.5
  * lon           (lon) float64 2kB 0.625 1.875 3.125 ... 356.9 358.1 359.4
  * year          (year) int64 520B 1950 1951 1952 1953 ... 2011 2012 2013 2014
    height        float64 8B 2.0
Data variables:
    temperature   (season, year, lat, lon) float32 54MB dask.array<chunksize=(1, 1, 180, 288), meta=np.ndarray>
    anomaly       (season, year, lat, lon) float32 54MB dask.array<chunksize=(1, 1, 180, 288), meta=np.ndarray>
    warming_rate  (season, lat, lon) float32 829kB dask.array<chunksize=(1, 180, 288), meta=np.ndarray>
Attributes:
    model:       GFDL-ESM4
    member:      r2i1p1f1
    experiment:  historical
    start_year:  1950
    end_year:    2014
    baseline:    1951-1980
Saved cmip6_processed.nc


In [2]:
# Global winter temperatures in 2000
climate.temperature.sel(season="DJF", year=2000)

# Global summer warming rates
climate.warming_rate.sel(season="JJA")

# Temperature anomalies in a geographic region
climate.anomaly.sel(
    season="MAM",
    year=1990,
    lat=slice(20, 60),
    lon=slice(230, 300)
)

<xarray.DataArray 'anomaly' (lat: 40, lon: 56)> Size: 9kB
dask.array<getitem, shape=(40, 56), dtype=float32, chunksize=(40, 56), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float64 320B 20.5 21.5 22.5 23.5 24.5 ... 56.5 57.5 58.5 59.5
  * lon      (lon) float64 448B 230.6 231.9 233.1 234.4 ... 296.9 298.1 299.4
    season   <U3 12B 'MAM'
    height   float64 8B 2.0
    year     int64 8B 1990
Attributes:
    cell_measures:  area: areacella
    cell_methods:   area: time: mean
    interp_method:  conserve_order2
    long_name:      Near-Surface Air Temperature
    original_name:  tas
    standard_name:  air_temperature
    units:          degC